# Historical Average Baseline
Same dataset, same feature set as `build_model_weather.ipynb` (LightGBM) and
`build_model_lr.ipynb` (Linear Regression) — first of the "simple" baselines
in the comparison chain.

This model doesn't learn anything from features in the ML sense. It just
memorizes the **average observed `travel_time`** for a group of similar
segments (e.g. same route, direction, stop, hour, weekday) from the training
data, and looks that average up at prediction time. It's the standard
"what usually happens" baseline that any learned model (LR, RF, XGBoost,
LightGBM) should be expected to beat — if a model can't beat this, it isn't
learning anything useful beyond historical averages.

**Split strategy note:** same as the other notebooks — random group-split by
`trip_id` for the current single-day dataset. Switch `USE_CHRONOLOGICAL_SPLIT`
to `True` once the dataset spans multiple months, and keep this setting
consistent across all baseline notebooks so results stay comparable.

In [24]:
import numpy as np
import polars as pl
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import joblib

In [25]:
DATA_PATH = "raw/processed_gtfs/baseline_dataset.parquet"

# Toggle this once the dataset spans multiple months (see note above).
# Requires a `service_date` column in df.
USE_CHRONOLOGICAL_SPLIT = False
CHRONOLOGICAL_SPLIT_DATE = "2026-06-01"  # train < this date, test >= this date

df = pl.read_parquet(DATA_PATH)
print(df.shape)

TARGET = "travel_time"
CATEGORICAL = ["route_id", "direction_id", "shape_id", "service_id", 'is_raining', 'is_snowing', 'is_fog', "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "has_major_event",]
NUMERIC = [
    "stop_sequence", "trip_progress",
    "hour", "weekday", "month", "is_peak",
    "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
    "stop_lat", "stop_lon", "latitude", "longitude", "bearing",
    'temperature_c', 'precipitation_mm', 'snowfall_cm', 'windspeed_kmh', 'weathercode',
    'segment_length',
    'scheduled_segment_speed_mps',
    'upstream_delay_seconds',
    'speed_mps',
    'headway_seconds',
    "ridership",
    "transfers",
]

FEATURES = CATEGORICAL + NUMERIC

(56764, 40)


## Split
Identical split logic to `build_model_lr.ipynb` / `build_model_weather.ipynb`
so all baselines are evaluated on the exact same train/test rows.

In [26]:
if USE_CHRONOLOGICAL_SPLIT:
    assert "service_date" in df.columns, "service_date column required for chronological split"

    train_df = df.filter(pl.col("service_date") < CHRONOLOGICAL_SPLIT_DATE)
    test_df = df.filter(pl.col("service_date") >= CHRONOLOGICAL_SPLIT_DATE)

    print(f"[chronological split @ {CHRONOLOGICAL_SPLIT_DATE}]")
    print(f"train: {train_df.shape} | test: {test_df.shape}")
    print(f"train trips: {train_df['trip_id'].n_unique():,} | "
          f"test trips: {test_df['trip_id'].n_unique():,} | "
          f"overlap: {len(set(train_df['trip_id']) & set(test_df['trip_id']))}")
else:
    groups = df["trip_id"].to_numpy()
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=groups))

    train_df = df[train_idx]
    test_df = df[test_idx]

    print("[random group split by trip_id]")
    print(f"train: {train_df.shape} | test: {test_df.shape}")
    print(f"train trips: {train_df['trip_id'].n_unique():,} | "
          f"test trips: {test_df['trip_id'].n_unique():,} | "
          f"overlap: {len(set(train_df['trip_id']) & set(test_df['trip_id']))}")

[random group split by trip_id]
train: (45363, 40) | test: (11401, 40)
train trips: 996 | test trips: 250 | overlap: 0


## Group-mean lookup table with fallback hierarchy
For each test row we want the average `travel_time` of the *most similar*
group of training rows. But at fine granularity, many exact combinations in
test won't have appeared (enough) in train — especially once the dataset
grows to 4 months and segment/hour combos get sparser per bucket. So we
build a **hierarchy of group keys**, from most to least specific, and fall
back down the hierarchy whenever a group wasn't seen (or was too small to
trust) in training:

1. `route_id, direction_id, stop_sequence, hour, weekday`  (most specific)
2. `route_id, direction_id, stop_sequence, hour`
3. `route_id, direction_id, stop_sequence`
4. `route_id, direction_id`
5. global mean  (always available, final fallback)

`MIN_GROUP_SIZE` controls how many training observations a group needs
before its average is considered trustworthy — small groups fall through to
the next, coarser level instead of overfitting to a couple of noisy rows.

In [27]:
MIN_GROUP_SIZE = 5

GROUP_LEVELS = [
    ["route_id", "direction_id", "stop_sequence", "hour", "weekday"],
    ["route_id", "direction_id", "stop_sequence", "hour"],
    ["route_id", "direction_id", "stop_sequence"],
    ["route_id", "direction_id"],
]

global_mean = train_df[TARGET].mean()
print(f"global mean travel_time: {global_mean:.2f} sec")

level_tables = []
for keys in GROUP_LEVELS:
    tbl = (
        train_df.group_by(keys)
        .agg([
            pl.col(TARGET).mean().alias("hist_avg"),
            pl.col(TARGET).count().alias("n_obs"),
        ])
        .filter(pl.col("n_obs") >= MIN_GROUP_SIZE)
        .drop("n_obs")
    )
    level_tables.append((keys, tbl))
    print(f"level {keys}: {tbl.height:,} groups with >= {MIN_GROUP_SIZE} obs")

global mean travel_time: 92.64 sec
level ['route_id', 'direction_id', 'stop_sequence', 'hour', 'weekday']: 3,601 groups with >= 5 obs
level ['route_id', 'direction_id', 'stop_sequence', 'hour']: 3,611 groups with >= 5 obs
level ['route_id', 'direction_id', 'stop_sequence']: 672 groups with >= 5 obs
level ['route_id', 'direction_id']: 10 groups with >= 5 obs


In [28]:
def predict_historical_avg(target_df: pl.DataFrame) -> np.ndarray:
    # Look up the historical average for each row, falling back to
    # coarser group keys (and finally the global mean) whenever a finer
    # group wasn't seen often enough in training.
    working = target_df.with_row_index("_row_idx").with_columns(
        pl.lit(None, dtype=pl.Float64).alias("hist_avg")
    )

    resolved = working.filter(pl.lit(False))  # empty, same schema
    remaining = working

    for keys, tbl in level_tables:
        if remaining.height == 0:
            break
        joined = remaining.drop("hist_avg").join(tbl, on=keys, how="left")
        matched = joined.filter(pl.col("hist_avg").is_not_null())
        unmatched = joined.filter(pl.col("hist_avg").is_null())

        resolved = pl.concat([resolved, matched], how="diagonal_relaxed")
        remaining = unmatched.drop("hist_avg").with_columns(
            pl.lit(None, dtype=pl.Float64).alias("hist_avg")
        )

    # anything left after every level falls back to the global mean
    if remaining.height > 0:
        remaining = remaining.with_columns(pl.lit(global_mean).alias("hist_avg"))
        resolved = pl.concat([resolved, remaining], how="diagonal_relaxed")

    resolved = resolved.sort("_row_idx")
    return resolved["hist_avg"].to_numpy()

preds_train = predict_historical_avg(train_df)
preds_test = predict_historical_avg(test_df)
print("predictions generated for train + test")

predictions generated for train + test


## Evaluate
Same metrics (MAE, RMSE, R²) and the same naive baseline comparison
(predicting `scheduled_segment_time` as-is) used in the other notebooks, so
all baselines are directly comparable.

In [29]:
y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

mae = mean_absolute_error(y_test, preds_test)
rmse = np.sqrt(mean_squared_error(y_test, preds_test))
r2 = r2_score(y_test, preds_test)

print(f"MAE:  {mae:.1f} sec")
print(f"RMSE: {rmse:.1f} sec")
print(f"R2:   {r2:.4f}")
mape = mean_absolute_percentage_error(y_test, preds_test)
wape = np.abs(y_test - preds_test).sum() / np.abs(y_test).sum()

print(f"MAPE: {mape*100:.1f}%  (caution: unstable if travel_time has near-zero rows)")
print(f"WAPE: {wape*100:.1f}%")
# naive baseline for comparison: predict the scheduled segment time as-is
naive_preds = test_df["scheduled_segment_time"].to_numpy()
naive_mae = mean_absolute_error(y_test, naive_preds)
print(f"\nNaive baseline (scheduled_segment_time only) MAE: {naive_mae:.1f} sec")
print(f"Improvement over naive: {(1 - mae / naive_mae) * 100:.1f}%")
naive_wape = np.abs(y_test - naive_preds).sum() / np.abs(y_test).sum()
print(f"Naive baseline WAPE: {naive_wape*100:.1f}%")

MAE:  38.4 sec
RMSE: 65.9 sec
R2:   0.3301
MAPE: 85.2%  (caution: unstable if travel_time has near-zero rows)
WAPE: 38.9%

Naive baseline (scheduled_segment_time only) MAE: 47.4 sec
Improvement over naive: 19.0%
Naive baseline WAPE: 48.0%


## Fallback-level diagnostics
Worth checking how many test rows were served by each level of the
hierarchy — if most rows fall all the way back to the global mean, the
group keys are too fine-grained (or training data too sparse) for this
baseline to be meaningful, and `MIN_GROUP_SIZE` / the key hierarchy should
be revisited once the 4-month dataset is available.

In [30]:
def level_usage(target_df: pl.DataFrame) -> pl.DataFrame:
    working = target_df.with_row_index("_row_idx")
    remaining = working
    rows = []
    for i, (keys, tbl) in enumerate(level_tables):
        if remaining.height == 0:
            rows.append({"level": i + 1, "keys": str(keys), "rows_matched": 0})
            continue
        joined = remaining.join(tbl, on=keys, how="left")
        matched = joined.filter(pl.col("hist_avg").is_not_null())
        remaining = joined.filter(pl.col("hist_avg").is_null()).drop("hist_avg")
        rows.append({"level": i + 1, "keys": str(keys), "rows_matched": matched.height})
    rows.append({"level": len(level_tables) + 1, "keys": "global_mean", "rows_matched": remaining.height})
    return pl.DataFrame(rows)

usage = level_usage(test_df)
usage = usage.with_columns((pl.col("rows_matched") / test_df.height * 100).round(1).alias("pct"))
print(usage)

shape: (5, 4)
┌───────┬─────────────────────────────────┬──────────────┬──────┐
│ level ┆ keys                            ┆ rows_matched ┆ pct  │
│ ---   ┆ ---                             ┆ ---          ┆ ---  │
│ i64   ┆ str                             ┆ i64          ┆ f64  │
╞═══════╪═════════════════════════════════╪══════════════╪══════╡
│ 1     ┆ ['route_id', 'direction_id', '… ┆ 3660         ┆ 32.1 │
│ 2     ┆ ['route_id', 'direction_id', '… ┆ 11           ┆ 0.1  │
│ 3     ┆ ['route_id', 'direction_id', '… ┆ 7730         ┆ 67.8 │
│ 4     ┆ ['route_id', 'direction_id']    ┆ 0            ┆ 0.0  │
│ 5     ┆ global_mean                     ┆ 0            ┆ 0.0  │
└───────┴─────────────────────────────────┴──────────────┴──────┘


## Save
Saved as a `.pkl` via joblib — the "model" here is just the ordered list of
group-key hierarchy tables plus the global mean fallback, everything
`predict_historical_avg` needs at inference time. Naming mirrors the other
baseline model files so all four are easy to find together in `models/`.

In [ ]:
artifact = {
    "level_tables": level_tables,
    "global_mean": global_mean,
    "min_group_size": MIN_GROUP_SIZE,
    "group_levels": GROUP_LEVELS,
}

# joblib.dump(artifact, "models/historical_avg_weather_segments_delay_and_ridership.pkl")
# print("model saved")

model saved
